# MovieLens 20M: Recommendation Systems and Clustering Analysis

## Project Overview

This project implements and evaluates various recommendation systems and clustering algorithms on the MovieLens 20M dataset.

### Algorithms Implemented:

**Recommendation Systems:**
1. **SVD (Traditional Matrix Factorization)** - Baseline collaborative filtering using Singular Value Decomposition
2. **PageRank** - Graph-based recommendation using PageRank algorithm
3. **Hybrid System** - Combination of SVD and PageRank
4. **ALS (Modern Matrix Factorization)** - Alternating Least Squares for implicit feedback
5. **ItemKNN (Traditional Collaborative Filtering)** - Item-based K-nearest neighbors

**Clustering Analysis:**
1. **K-means Clustering** - Partition-based clustering
2. **Hierarchical Clustering** - Agglomerative clustering
3. **PCA (Principal Component Analysis)** - Dimensionality reduction and its impact on clustering

### Evaluation Metrics:

**Rating Prediction:**
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)

**Ranking Quality:**
- Precision@K
- Recall@K
- NDCG@K (Normalized Discounted Cumulative Gain)
- HitRate@K

**Clustering Quality:**
- Silhouette Score
- Davies-Bouldin Index
- Calinski-Harabasz Index

## 1. Import Libraries and Setup

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import MovieLensLoader
from src.recommender import (SVDRecommender, PageRankRecommender, HybridRecommender,
                             ALSRecommender, ItemKNNRecommender)
from src.clustering import MovieClusterer, DimensionalityReducer
from src.visualization import Visualizer
from src.evaluation import RecommenderEvaluator, create_relevance_set

print("Libraries imported successfully!")

## 2. Data Loading

Load the MovieLens 20M dataset. This dataset contains:
- 20 million ratings
- 138,000 users
- 27,000 movies

For faster processing, we'll use a 2 million rating sample.

In [ ]:
loader = MovieLensLoader(data_dir='./data')

try:
    loader.load_data(sample_size=2000000)
    print(f"\nDataset loaded successfully!")
    print(f"Ratings: {len(loader.ratings):,}")
    print(f"Users: {loader.ratings['userId'].nunique():,}")
    print(f"Movies: {loader.ratings['movieId'].nunique():,}")
except FileNotFoundError:
    print("\nError: Data files not found!")
    print("Please download MovieLens 20M dataset from Kaggle")
    print("and extract to data/ml-20m/ directory")

## 3. Data Preprocessing

Filter the dataset to include only:
- Users with at least 50 ratings
- Movies with at least 50 ratings

This ensures better quality recommendations.

In [ ]:
user_movie_matrix, filtered_ratings = loader.preprocess_for_recommendation(
    min_user_ratings=50,
    min_movie_ratings=50
)

print(f"\nFiltered dataset:")
print(f"Matrix shape: {user_movie_matrix.shape}")
print(f"Total ratings: {len(filtered_ratings):,}")
print(f"Sparsity: {(1 - filtered_ratings.shape[0] / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1])) * 100:.2f}%")

## 4. Train-Test Split

Split data using stratified sampling:
- For each user, randomly select 20% of ratings for testing
- Remaining 80% for training
- Ensures all users appear in both train and test sets

In [ ]:
print("Splitting data into train and test sets...")

train_list = []
test_list = []

for user_id in filtered_ratings['userId'].unique():
    user_ratings = filtered_ratings[filtered_ratings['userId'] == user_id]
    n_test = max(1, int(len(user_ratings) * 0.2))
    
    user_ratings = user_ratings.sample(frac=1, random_state=42)
    
    test_list.append(user_ratings.iloc[:n_test])
    train_list.append(user_ratings.iloc[n_test:])

train_ratings = pd.concat(train_list, ignore_index=True)
test_ratings = pd.concat(test_list, ignore_index=True)

print(f"Train set: {len(train_ratings):,} ratings")
print(f"Test set: {len(test_ratings):,} ratings")

train_matrix = train_ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating',
    fill_value=0
)

print(f"Train matrix shape: {train_matrix.shape}")

## 5. Helper Function: Evaluate Ranking Quality

This function evaluates recommendation ranking quality using:
- Precision@K: Proportion of relevant items in top-K
- Recall@K: Proportion of relevant items retrieved
- NDCG@K: Normalized discounted cumulative gain
- HitRate@K: Proportion of users with at least one relevant item in top-K

In [ ]:
def evaluate_ranking_quality(recommender, test_ratings, k=10, sample_users=500):
    """
    Evaluate recommendation ranking quality
    
    Args:
        recommender: Recommender object
        test_ratings: Test set DataFrame
        k: Number of recommendations
        sample_users: Number of users to sample for evaluation
    
    Returns:
        Dictionary of ranking metrics
    """
    user_relevant = create_relevance_set(test_ratings, threshold=4.0)
    sampled_users = list(user_relevant.keys())[:sample_users]
    
    user_recommendations = {}
    for user_id in sampled_users:
        try:
            recs = recommender.recommend_for_user(user_id, top_n=k, exclude_rated=True)
            user_recommendations[user_id] = [movie_id for movie_id, _ in recs]
        except:
            continue
    
    evaluator = RecommenderEvaluator(k=k)
    metrics = evaluator.evaluate_recommendations(
        user_recommendations,
        {uid: user_relevant[uid] for uid in user_recommendations if uid in user_relevant},
        k=k
    )
    
    print(f"  Precision@{k}: {metrics[f'Precision@{k}']:.4f}")
    print(f"  Recall@{k}: {metrics[f'Recall@{k}']:.4f}")
    print(f"  NDCG@{k}: {metrics[f'NDCG@{k}']:.4f}")
    print(f"  HitRate@{k}: {metrics[f'HitRate@{k}']:.4f}")
    
    return metrics

print("Helper function defined!")

## 6. SVD Recommender (Traditional Matrix Factorization)

SVD decomposes the user-movie matrix into:
- User factors (U)
- Singular values (Σ)
- Movie factors (V^T)

Approximation: R ≈ U × Σ × V^T

In [ ]:
print("\n" + "="*80)
print("Training SVD Recommender (Traditional Matrix Factorization)")
print("="*80)

svd_recommender = SVDRecommender(n_components=200)
svd_recommender.fit(train_matrix)
svd_metrics = svd_recommender.evaluate(test_ratings)

print(f"\nRating Prediction Metrics:")
print(f"  MAE: {svd_metrics['MAE']:.4f}")
print(f"  RMSE: {svd_metrics['RMSE']:.4f}")

print(f"\nRanking Quality Metrics:")
svd_ranking = evaluate_ranking_quality(svd_recommender, test_ratings, k=10, sample_users=500)

## 7. PageRank Recommender (Graph-based)

PageRank-based recommendation:
- Constructs user-movie bipartite graph
- Applies PageRank algorithm to rank movies
- Combines PageRank scores with collaborative filtering
- Captures global popularity and local preferences

**Optimization**: Uses vectorized operations for fast recommendation generation.

In [ ]:
print("\n" + "="*80)
print("Training PageRank Recommender (Graph-based)")
print("="*80)

pagerank_recommender = PageRankRecommender(alpha=0.85, cf_weight=0.5)
pagerank_recommender.fit(train_matrix)
pr_metrics = pagerank_recommender.evaluate(test_ratings)

print(f"\nRating Prediction Metrics:")
print(f"  MAE: {pr_metrics['MAE']:.4f}")
print(f"  RMSE: {pr_metrics['RMSE']:.4f}")

print(f"\nRanking Quality Metrics:")
pr_ranking = evaluate_ranking_quality(pagerank_recommender, test_ratings, k=10, sample_users=500)

## 8. Hybrid Recommender (SVD + PageRank)

Combines the strengths of both approaches:
- SVD: Captures latent user preferences
- PageRank: Captures item popularity and graph structure
- Weighted combination of predictions

In [ ]:
print("\n" + "="*80)
print("Training Hybrid Recommender (SVD + PageRank)")
print("="*80)

hybrid_recommender = HybridRecommender(
    svd_recommender=svd_recommender,
    pagerank_recommender=pagerank_recommender,
    svd_weight=0.5
)
hybrid_metrics = hybrid_recommender.evaluate(test_ratings)

print(f"\nRating Prediction Metrics:")
print(f"  MAE: {hybrid_metrics['MAE']:.4f}")
print(f"  RMSE: {hybrid_metrics['RMSE']:.4f}")

print(f"\nRanking Quality Metrics:")
hybrid_ranking = evaluate_ranking_quality(hybrid_recommender, test_ratings, k=10, sample_users=500)

## 9. ALS Recommender (Modern Matrix Factorization)

Alternating Least Squares (ALS) is an iterative optimization algorithm that:
- Alternately fixes user factors and optimizes movie factors
- Then fixes movie factors and optimizes user factors
- Handles implicit feedback better than SVD
- Used by Netflix, Spotify, and YouTube

In [ ]:
print("\n" + "="*80)
print("Training ALS Recommender (Modern Matrix Factorization)")
print("="*80)

als_recommender = ALSRecommender(n_factors=100, n_iterations=10, regularization=0.01)
als_recommender.fit(train_matrix)
als_metrics = als_recommender.evaluate(test_ratings)

print(f"\nRating Prediction Metrics:")
print(f"  MAE: {als_metrics['MAE']:.4f}")
print(f"  RMSE: {als_metrics['RMSE']:.4f}")

print(f"\nRanking Quality Metrics:")
als_ranking = evaluate_ranking_quality(als_recommender, test_ratings, k=10, sample_users=500)

## 10. ItemKNN Recommender (Traditional Collaborative Filtering)

Item-based K-Nearest Neighbors:
- Computes item-item similarity matrix
- Predicts ratings based on similar items the user has rated
- Classic approach used by Amazon in early days

In [ ]:
print("\n" + "="*80)
print("Training ItemKNN Recommender (Traditional Collaborative Filtering)")
print("="*80)

itemknn_recommender = ItemKNNRecommender(k=30, similarity_metric='cosine')
itemknn_recommender.fit(train_matrix)
itemknn_metrics = itemknn_recommender.evaluate(test_ratings)

print(f"\nRating Prediction Metrics:")
print(f"  MAE: {itemknn_metrics['MAE']:.4f}")
print(f"  RMSE: {itemknn_metrics['RMSE']:.4f}")

print(f"\nRanking Quality Metrics:")
itemknn_ranking = evaluate_ranking_quality(itemknn_recommender, test_ratings, k=10, sample_users=500)

## 11. Performance Comparison: All Recommenders

Compare all 5 recommendation systems across two dimensions:
1. **Rating Prediction** (MAE, RMSE) - Lower is better
2. **Ranking Quality** (Precision, Recall, NDCG, HitRate) - Higher is better

In [ ]:
print("\n\n" + "="*100)
print("COMPREHENSIVE PERFORMANCE COMPARISON")
print("="*100)

print("\n[Rating Prediction Metrics] (Lower is Better)")
print("-"*100)
rating_comparison = pd.DataFrame({
    'Model': ['SVD (Traditional MF)', 'PageRank', 'Hybrid', 'ALS (Modern MF)', 'ItemKNN (Traditional CF)'],
    'MAE': [svd_metrics['MAE'], pr_metrics['MAE'], hybrid_metrics['MAE'],
            als_metrics['MAE'], itemknn_metrics['MAE']],
    'RMSE': [svd_metrics['RMSE'], pr_metrics['RMSE'], hybrid_metrics['RMSE'],
             als_metrics['RMSE'], itemknn_metrics['RMSE']]
})
print(rating_comparison.to_string(index=False, float_format='%.4f'))

print("\n\n[Ranking Quality Metrics] (Higher is Better)")
print("-"*100)
ranking_comparison = pd.DataFrame({
    'Model': ['SVD (Traditional MF)', 'PageRank', 'Hybrid', 'ALS (Modern MF)', 'ItemKNN (Traditional CF)'],
    'Precision@10': [svd_ranking['Precision@10'], pr_ranking['Precision@10'], hybrid_ranking['Precision@10'],
                    als_ranking['Precision@10'], itemknn_ranking['Precision@10']],
    'Recall@10': [svd_ranking['Recall@10'], pr_ranking['Recall@10'], hybrid_ranking['Recall@10'],
                 als_ranking['Recall@10'], itemknn_ranking['Recall@10']],
    'NDCG@10': [svd_ranking['NDCG@10'], pr_ranking['NDCG@10'], hybrid_ranking['NDCG@10'],
               als_ranking['NDCG@10'], itemknn_ranking['NDCG@10']],
    'HitRate@10': [svd_ranking['HitRate@10'], pr_ranking['HitRate@10'], hybrid_ranking['HitRate@10'],
                  als_ranking['HitRate@10'], itemknn_ranking['HitRate@10']]
})
print(ranking_comparison.to_string(index=False, float_format='%.4f'))
print("="*100)

## 12. Visualization: Recommender Comparison

In [ ]:
plt.figure(figsize=(14, 6))
models = ['SVD\n(Traditional MF)', 'PageRank', 'Hybrid', 'ALS\n(Modern MF)', 'ItemKNN\n(Traditional CF)']
mae_values = [svd_metrics['MAE'], pr_metrics['MAE'], hybrid_metrics['MAE'],
              als_metrics['MAE'], itemknn_metrics['MAE']]
rmse_values = [svd_metrics['RMSE'], pr_metrics['RMSE'], hybrid_metrics['RMSE'],
               als_metrics['RMSE'], itemknn_metrics['RMSE']]

x = np.arange(len(models))
width = 0.35

plt.bar(x - width/2, mae_values, width, label='MAE', alpha=0.8, color='#2E86AB')
plt.bar(x + width/2, rmse_values, width, label='RMSE', alpha=0.8, color='#A23B72')

plt.xlabel('Recommendation System', fontsize=12)
plt.ylabel('Error', fontsize=12)
plt.title('Recommendation System Performance Comparison', fontsize=14, fontweight='bold')
plt.xticks(x, models)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/recommender_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved to ./figures/recommender_comparison.png")

## 13. Sample Recommendations

Generate top-5 movie recommendations for a sample user using each algorithm

In [ ]:
print("\n" + "="*80)
print("SAMPLE RECOMMENDATIONS COMPARISON")
print("="*80)

sample_user = train_matrix.index[10]
print(f"\nGenerating recommendations for User {sample_user}:\n")

recommenders = [
    ('SVD (Traditional MF)', svd_recommender),
    ('PageRank', pagerank_recommender),
    ('Hybrid', hybrid_recommender),
    ('ALS (Modern MF)', als_recommender),
    ('ItemKNN (Traditional CF)', itemknn_recommender)
]

for name, recommender in recommenders:
    recs = recommender.recommend_for_user(sample_user, top_n=5)
    print(f"{name}:")
    for i, (movie_id, score) in enumerate(recs, 1):
        movie_info = loader.get_movie_info([movie_id])
        if len(movie_info) > 0:
            title = movie_info.iloc[0]['title']
            print(f"  {i}. {title} (Score: {score:.4f})")
    print()

## 14. Movie Clustering: K-means

Cluster movies based on genre features using K-means algorithm

In [ ]:
print("\n" + "="*80)
print("MOVIE CLUSTERING ANALYSIS")
print("="*80)

movie_features = loader.create_movie_features(use_genome=False)
print(f"\nClustering {movie_features.shape[0]} movies with {movie_features.shape[1]} features")

clusterer_kmeans = MovieClusterer()
labels_kmeans = clusterer_kmeans.kmeans_clustering(movie_features, n_clusters=10)

## 15. Movie Clustering: Hierarchical

Apply hierarchical clustering using Ward linkage

In [ ]:
clusterer_hierarchical = MovieClusterer()
labels_hierarchical = clusterer_hierarchical.hierarchical_clustering(
    movie_features, n_clusters=10
)

## 16. Clustering Performance Comparison

In [ ]:
print("\n" + "="*80)
print("CLUSTERING METHOD COMPARISON")
print("="*80)

kmeans_metrics = MovieClusterer.evaluate_clustering(movie_features, labels_kmeans)
hierarchical_metrics = MovieClusterer.evaluate_clustering(movie_features, labels_hierarchical)

comparison_df = pd.DataFrame({
    'Method': ['K-means', 'Hierarchical'],
    'Silhouette Score': [kmeans_metrics['silhouette'], hierarchical_metrics['silhouette']],
    'Davies-Bouldin Index': [kmeans_metrics['davies_bouldin'], hierarchical_metrics['davies_bouldin']],
    'Calinski-Harabasz Index': [kmeans_metrics['calinski_harabasz'], hierarchical_metrics['calinski_harabasz']]
})

print("\n" + comparison_df.to_string(index=False))

viz = Visualizer(save_dir='./figures')
viz.plot_clustering_comparison({
    'K-means': kmeans_metrics,
    'Hierarchical': hierarchical_metrics
})

## 17. Cluster Statistics and Distribution

In [ ]:
print("\nK-means Cluster Statistics:")
cluster_stats = clusterer_kmeans.get_cluster_statistics(loader.movies)
print(cluster_stats.to_string())

viz.plot_cluster_distribution(labels_kmeans, 'kmeans_distribution.png')
viz.plot_pca_2d_clusters(movie_features, labels_kmeans, 'kmeans_2d_visualization.png')

## 18. Finding Optimal K Value

Use elbow method and silhouette analysis to find optimal number of clusters

In [ ]:
print("\n" + "="*80)
print("FINDING OPTIMAL K VALUE")
print("="*80)

optimal_k_results = clusterer_kmeans.find_optimal_k(
    movie_features, k_range=range(5, 21)
)

viz.plot_elbow_curve(
    list(optimal_k_results.keys()),
    optimal_k_results,
    metric_name='silhouette',
    save_name='silhouette_elbow.png'
)

viz.plot_elbow_curve(
    list(optimal_k_results.keys()),
    optimal_k_results,
    metric_name='davies_bouldin',
    save_name='davies_bouldin_elbow.png'
)

## 19. Dimensionality Reduction with PCA

Apply Principal Component Analysis to reduce feature dimensions

In [ ]:
print("\n" + "="*80)
print("PCA DIMENSIONALITY REDUCTION ANALYSIS")
print("="*80)

reducer = DimensionalityReducer(n_components=20)

comparison_results = reducer.compare_clustering_with_without_pca(
    movie_features,
    n_clusters=10,
    method='kmeans'
)

viz.plot_pca_variance(comparison_results['pca'])

## 20. PCA Impact on Clustering Quality

In [ ]:
viz.plot_multi_metrics_comparison(comparison_results)

viz.plot_pca_2d_clusters(
    comparison_results['reduced']['features'],
    comparison_results['reduced']['labels'],
    'pca_reduced_clusters_2d.png'
)

## 21. Summary and Key Findings

In [ ]:
print("\n\n" + "="*80)
print("ANALYSIS COMPLETE - KEY FINDINGS")
print("="*80)

print("\n1. RECOMMENDATION SYSTEM PERFORMANCE:")
best_model = rating_comparison.loc[rating_comparison['MAE'].idxmin()]
print(f"   Best Model: {best_model['Model']}")
print(f"   MAE: {best_model['MAE']:.4f}")
print(f"   RMSE: {best_model['RMSE']:.4f}")

print("\n2. CLUSTERING ALGORITHM COMPARISON:")
print(f"   K-means Silhouette Score: {kmeans_metrics['silhouette']:.4f}")
print(f"   Hierarchical Silhouette Score: {hierarchical_metrics['silhouette']:.4f}")
better_clustering = 'K-means' if kmeans_metrics['silhouette'] > hierarchical_metrics['silhouette'] else 'Hierarchical'
print(f"   Better Method: {better_clustering}")

print("\n3. DIMENSIONALITY REDUCTION IMPACT:")
original_silhouette = comparison_results['original']['metrics']['silhouette']
reduced_silhouette = comparison_results['reduced']['metrics']['silhouette']
print(f"   Original Features Silhouette: {original_silhouette:.4f}")
print(f"   PCA Reduced Silhouette: {reduced_silhouette:.4f}")
if reduced_silhouette > original_silhouette:
    print("   Conclusion: PCA improves clustering quality")
else:
    print("   Conclusion: Original features perform better")

print("\n4. VISUALIZATIONS:")
print("   All visualizations saved to ./figures/ directory")

print("\n" + "="*80)
print("Analysis completed successfully!")
print("="*80)